In [ ]:
""" Results obtained from the following IPython notebooks:

- Gopi_TEC_Brasilia_analysis_09_27_2024.ipynb (BRAZ station)
- Gopi_TEC_Brasilia_analysis_Dec_2024.ipynb (BRAZ station)
- Gopi_TEC_Cuiaba_analysis_09_27_2024.ipynb (CUIB station)
- Gopi_TEC_Cuiaba_analysis_Dec_2024.ipynb (CUIB station)
- Gopi_TEC_Manaus_analysis_09_27_2024.ipynb (NAUS station)
- Gopi_TEC_Manaus_analysis_Dec_2024.ipynb (NAUS station)
- Gopi_TEC_Porto_Alegre_analysis_09_27_2024.ipynb (POAL station)
- Gopi_TEC_Porto_Alegre_analysis_Dec_2024.ipynb (POAL station)
- Gopi_TEC_SJC_analysis_09_27_2024.ipynb (SJSP station)
- Gopi_TEC_SJC_analysis_Dec_2024.ipynb (SJSP station)
- Gopi_TEC_Salvador_analysis_09_27_2024.ipynb (SAVO station)
- Gopi_TEC_Salvador_analysis_Dec_2024.ipynb (SAVO station)
- Gopi_TEC_Sao_Luis_analysis_09_27_2024.ipynb (SALU station)
- Gopi_TEC_Sao_Luis_analysis_Dec_2024.ipynb (SALU station) """

# Functions

In [1]:
import numpy as np
from scipy import stats
import pandas as pd

def pearson_correlation(observed, predicted):

    correlation, p_value = stats.pearsonr(observed, predicted)
    return correlation, p_value

def taylor_skill_score(observed, predicted):
    observed = np.array(observed)
    predicted = np.array(predicted)

    correlation = np.corrcoef(observed, predicted)[0, 1]

    std_obs = np.std(observed, ddof=1)
    std_pred = np.std(predicted, ddof=1)

    if std_obs == 0 or std_pred == 0:
        return 0.0

    std_ratio = std_pred / std_obs

    denominator = (std_ratio + 1/std_ratio)**2 * (1 + 1.0)
    skill_score = (4 * (1 + correlation)) / denominator

    return skill_score

def kling_gupta_efficiency(observed, predicted):
    observed = np.array(observed)
    predicted = np.array(predicted)

    r = np.corrcoef(observed, predicted)[0, 1]

    std_obs = np.std(observed, ddof=1)
    std_pred = np.std(predicted, ddof=1)
    alpha = std_pred / std_obs if std_obs != 0 else np.inf

    mean_obs = np.mean(observed)
    mean_pred = np.mean(predicted)
    beta = mean_pred / mean_obs if mean_obs != 0 else np.inf

    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

    components = {'r': r, 'alpha': alpha, 'beta': beta}

    return kge, components

def calculate_all_metrics(observed, predicted, model_name="Model"):

    observed = np.array(observed)
    predicted = np.array(predicted)

    if len(observed) < 2 or len(predicted) < 2:
        return {
            'model': model_name,
            'mae': np.nan,
            'rmse': np.nan,
            'correlation': np.nan,
            'correlation_p_value': np.nan,
            'taylor_skill_score': np.nan,
            'kge': np.nan,
            'kge_r': np.nan,
            'kge_alpha': np.nan,
            'kge_beta': np.nan
        }

    mask = ~(np.isnan(observed) | np.isnan(predicted))
    observed_filtered = observed[mask]
    predicted_filtered = predicted[mask]

    if len(observed_filtered) < 2:
        return {
            'model': model_name,
            'mae': np.nan,
            'rmse': np.nan,
            'correlation': np.nan,
            'correlation_p_value': np.nan,
            'taylor_skill_score': np.nan,
            'kge': np.nan,
            'kge_r': np.nan,
            'kge_alpha': np.nan,
            'kge_beta': np.nan
        }

    diff = observed_filtered - predicted_filtered
    mae = np.mean(np.abs(diff))
    rmse = np.sqrt(np.mean(diff**2))

    correlation, p_value = pearson_correlation(observed_filtered, predicted_filtered)

    tss = taylor_skill_score(observed_filtered, predicted_filtered)

    kge, kge_components = kling_gupta_efficiency(observed_filtered, predicted_filtered)

    metrics = {
        'model': model_name,
        'mae': mae,
        'rmse': rmse,
        'correlation': correlation,
        'correlation_p_value': p_value,
        'taylor_skill_score': tss,
        'kge': kge,
        'kge_r': kge_components['r'],
        'kge_alpha': kge_components['alpha'],
        'kge_beta': kge_components['beta']
    }

    return metrics

def enhanced_analysis(data, reference_model='Gopi'):

    model_data_pairs = {}

    first_station = list(data.keys())[0]
    available_models = [model for model in data[first_station].keys() if model != reference_model]

    for model in available_models:
        model_data_pairs[model] = {
            'reference': [],
            'model': []
        }

    for station in data.keys():
        station_ref_data = data[station][reference_model]

        for model in available_models:
            model_data = data[station][model]

            for i in range(min(len(model_data), len(station_ref_data))):
                if not np.isnan(model_data[i]) and not np.isnan(station_ref_data[i]):
                    model_data_pairs[model]['reference'].append(station_ref_data[i])
                    model_data_pairs[model]['model'].append(model_data[i])

    all_metrics = {}
    for model in available_models:
        reference_values = np.array(model_data_pairs[model]['reference'])
        model_values = np.array(model_data_pairs[model]['model'])

        if len(reference_values) > 1:
            metrics = calculate_all_metrics(reference_values, model_values, model)
        else:

            metrics = {
                'model': model,
                'mae': np.nan,
                'rmse': np.nan,
                'correlation': np.nan,
                'correlation_p_value': np.nan,
                'taylor_skill_score': np.nan,
                'kge': np.nan,
                'kge_r': np.nan,
                'kge_alpha': np.nan,
                'kge_beta': np.nan
            }
        all_metrics[model] = metrics

    return all_metrics

def print_enhanced_results(results, title="Enhanced Analysis Results"):
    print("=" * 80)
    print(f"{title}")
    print("=" * 80)

    sorted_models = list(results.keys())

    for model in sorted_models:
        metrics = results[model]
        print(f"\n{model.upper()}:")
        print(f"  MAE:                   {metrics['mae']:.2f}")
        print(f"  RMSE:                  {metrics['rmse']:.2f}")
        print(f"  Pearson Correlation:   {metrics['correlation']:.3f} (p={metrics['correlation_p_value']:.3f})")
        print(f"  Taylor Skill Score:    {metrics['taylor_skill_score']:.3f}")
        print(f"  Kling-Gupta Efficiency: {metrics['kge']:.3f}")
        print(f"    - Correlation (r):   {metrics['kge_r']:.3f}")
        print(f"    - Alpha (σp/σo):     {metrics['kge_alpha']:.3f}")
        print(f"    - Beta (μp/μo):      {metrics['kge_beta']:.3f}")


# September

## September 27, 2024: 00:50 UT, 01:00 UT and 01:10 UT

In [ ]:
#The values ​​below refer, for each TEC source, consecutively, to the following times for September 27, 2024: 00:50 UT, 01:00 UT and 01:10 UT.

In [ ]:
import numpy as np

data = {
    'BRAZ': {
        'Gopi': [91.93, 92.28, 93.38],
        'EMBRACE': [72.52, 71.86, 67.75],
        'MAGGIA': [138.74, 133.04, 129.11]
    },
    'CUIB': {
        'Gopi': [73.90, 73.87, 74.22],
        'EMBRACE': [36.89, 43.34, 42.93],
        'MAGGIA': [75.94, 111.90, 85.23]
    },
    'NAUS': {
        'Gopi': [19.83, 19.68, 19.47],
        'EMBRACE': [28.84, 32.11, 34.07],
        'MAGGIA': [40.93, 39.25, 44.92]
    },
    'POAL': {
        'Gopi': [14.77, 15.80, 16.92],
        'EMBRACE': [33.90, 33.01, 34.13],
        'MAGGIA': [29.80, 27.86, 26.85]
    },
    'SALU': {
        'Gopi': [21.98, 20.75, 19.45],
        'EMBRACE': [39.31, 41.23, 34.59],
        'MAGGIA': [38.69, 42.25, 43.50]
    },
    'SAVO': {
        'Gopi': [101.46, 98.65, 94.63],
        'EMBRACE': [69.32, 73.81, 73.59],
        'MAGGIA': [121.90, 133.17, 127.19]
    },
    'SJSP': {
        'Gopi': [30.89, 30.06, 29.8],
        'EMBRACE': [31.64, 30.80, 28.02],
        'MAGGIA': [51.12, 49.78, 47.96]
    }
}

gopi_all = []
embrace_all = []
maggia_all = []

for station in data.keys():
    gopi_all.extend(data[station]['Gopi'])
    embrace_all.extend(data[station]['EMBRACE'])
    maggia_all.extend(data[station]['MAGGIA'])

gopi_all = np.array(gopi_all)
embrace_all = np.array(embrace_all)
maggia_all = np.array(maggia_all)

diff_embrace = gopi_all - embrace_all
diff_maggia = gopi_all - maggia_all

mae_embrace = np.mean(np.abs(diff_embrace))
mae_maggia = np.mean(np.abs(diff_maggia))

rmse_embrace = np.sqrt(np.mean(diff_embrace**2))
rmse_maggia = np.sqrt(np.mean(diff_maggia**2))

print("="*60)
print("CONSOLIDATED RESULTS - September 27, 2024")
print("(21 measurements: 7 stations × 3 times)")
print("="*60)
print(f"GOPI vs EMBRACE:")
print(f"  MAE:  {mae_embrace:.2f}")
print(f"  RMSE: {rmse_embrace:.2f}")
print()
print(f"GOPI vs MAGGIA:")
print(f"  MAE:  {mae_maggia:.2f}")
print(f"  RMSE: {rmse_maggia:.2f}")

results_september = enhanced_analysis(data)
print_enhanced_results(results_september, "COMPREHENSIVE ANALYSIS - September 27, 2024")

CONSOLIDATED RESULTS - September 27, 2024
(21 measurements: 7 stations × 3 times)
GOPI vs EMBRACE:
  MAE:  18.48
  RMSE: 20.91

GOPI vs MAGGIA:
  MAE:  23.11
  RMSE: 25.60
COMPREHENSIVE ANALYSIS - September 27, 2024

EMBRACE:
  MAE:                   18.48
  RMSE:                  20.91
  Pearson Correlation:   0.889 (p=0.000)
  Taylor Skill Score:    0.606
  Kling-Gupta Efficiency: 0.480
    - Correlation (r):   0.889
    - Alpha (σp/σo):     0.501
    - Beta (μp/μo):      0.905

MAGGIA:
  MAE:                   23.11
  RMSE:                  25.60
  Pearson Correlation:   0.974 (p=0.000)
  Taylor Skill Score:    0.953
  Kling-Gupta Efficiency: 0.494
    - Correlation (r):   0.974
    - Alpha (σp/σo):     1.209
    - Beta (μp/μo):      1.461


# December

## December 1, 18, 19 and 22, 2024: 18:00 UT

In [ ]:
#The values ​​below refer, for each TEC source, consecutively, to the following times for December 1, 18, 19 and 22, 2024: 18:00 UT.

In [2]:
import numpy as np

data = {
    'BRAZ': {
        'Gopi': [71.17, 79.74, 72.14, 78.26],
        'EMBRACE': [71.76, 81.36, 61.01, 76.91],
        'IGS': [94.58, 93.76, 83.53, 92.26],
        'MAGGIA': [85.46, 99.78, 80.46, 94.35],
        'Nagoya': [79.03, 82.50, 67.53, 75.63]
    },
    'CUIB': {
        'Gopi': [74.68, 74.03, 68.55, 75.19],
        'EMBRACE': [65.60, 76.98, 65.20, 68.62],
        'IGS': [95.92, 90.58, 82.04, 91.20],
        'MAGGIA': [100.93, 94.45, 82.78, 94.83],
        'Nagoya': [65.51, 75.42, 66.21, 70.67]
    },
    'NAUS': {
        'Gopi': [79.52, 67.33, 62.93, 66.96],
        'EMBRACE': [79.02, 63.07, 62.12, 67.53],
        'IGS': [94.34, 81.60, 78.54, 82.80],
        'MAGGIA': [100.49, 87.40, 82.94, 93.03],
        'Nagoya': [np.nan, 61.57, 64.89, 66.47]
    },
    'POAL': {
        'Gopi': [59.27, 58.40, 61.57, 60.67],
        'EMBRACE': [74.77, 75.83, 73.31, 75.73],
        'IGS': [81.85, 75.19, 78.20, 78.85],
        'MAGGIA': [84.14, 75.03, 80.20, 82.55],
        'Nagoya': [75.83, 74.34, 73.45, 76.95]
    },
    'SALU': {
        'Gopi': [74.54, 64.24, 57.14, 60.04],
        'EMBRACE': [80.89, 66.44, 59.13, 61.26],
        'IGS': [94.86, 75.58, 72.76, 76.84],
        'MAGGIA': [98.98, 79.03, 72.46, 81.07],
        'Nagoya': [58.45, 66.20, 59.06, 62.33]
    },
    'SAVO': {
        'Gopi': [73.75, 90.05, 72.09, 86.30],
        'EMBRACE': [76.41, 83.04, 75.58, 74.56],
        'IGS': [94.69, 93.59, 82.56, 92.33],
        'MAGGIA': [95.67, 99.74, 84.99, 99.28],
        'Nagoya': [np.nan, 83.50, 70.32, 72.88] # in the record of this line that has the value np.nan, the value is, in fact, -8.12, but a negative TEC value does not exist.
    },
    'SJSP': {
        'Gopi': [67.11, 71.73, 67.35, 67.49],
        'EMBRACE': [78.24, 80.30, 76.98, 78.06],
        'IGS': [88.07, 85.40, 80.77, 85.00],
        'MAGGIA': [88.18, 83.34, 80.46, 87.43],
        'Nagoya': [78.57, 83.00, 76.41, 80.96]
    }
}

gopi_embrace = []
embrace_all = []

gopi_igs = []
igs_all = []

gopi_maggia = []
maggia_all = []

gopi_nagoya = []
nagoya_all = []

for station in data.keys():
    for i in range(len(data[station]['Gopi'])):
        gopi_val = data[station]['Gopi'][i]

        if i < len(data[station]['EMBRACE']) and not np.isnan(gopi_val) and not np.isnan(data[station]['EMBRACE'][i]):
            gopi_embrace.append(gopi_val)
            embrace_all.append(data[station]['EMBRACE'][i])

        if i < len(data[station]['IGS']) and not np.isnan(gopi_val) and not np.isnan(data[station]['IGS'][i]):
            gopi_igs.append(gopi_val)
            igs_all.append(data[station]['IGS'][i])

        if i < len(data[station]['MAGGIA']) and not np.isnan(gopi_val) and not np.isnan(data[station]['MAGGIA'][i]):
            gopi_maggia.append(gopi_val)
            maggia_all.append(data[station]['MAGGIA'][i])

        if i < len(data[station]['Nagoya']) and not np.isnan(gopi_val) and not np.isnan(data[station]['Nagoya'][i]):
            gopi_nagoya.append(gopi_val)
            nagoya_all.append(data[station]['Nagoya'][i])

gopi_embrace = np.array(gopi_embrace)
embrace_all = np.array(embrace_all)

gopi_igs = np.array(gopi_igs)
igs_all = np.array(igs_all)

gopi_maggia = np.array(gopi_maggia)
maggia_all = np.array(maggia_all)

gopi_nagoya = np.array(gopi_nagoya)
nagoya_all = np.array(nagoya_all)

diff_embrace = gopi_embrace - embrace_all
diff_igs = gopi_igs - igs_all
diff_maggia = gopi_maggia - maggia_all
diff_nagoya = gopi_nagoya - nagoya_all

mae_embrace = np.mean(np.abs(diff_embrace))
rmse_embrace = np.sqrt(np.mean(diff_embrace**2))

mae_igs = np.mean(np.abs(diff_igs))
rmse_igs = np.sqrt(np.mean(diff_igs**2))

mae_maggia = np.mean(np.abs(diff_maggia))
rmse_maggia = np.sqrt(np.mean(diff_maggia**2))

mae_nagoya = np.mean(np.abs(diff_nagoya))
rmse_nagoya = np.sqrt(np.mean(diff_nagoya**2))

print("="*70)
print("CONSOLIDATED RESULTS - December 1, 18, 19 and 22, 2024")
print("(28 measurements: 7 stations × 4 times)")
print("="*70)
print(f"GOPI vs EMBRACE:")
print(f"  MAE:  {mae_embrace:.2f}")
print(f"  RMSE: {rmse_embrace:.2f}")
print()
print(f"GOPI vs IGS:")
print(f"  MAE:  {mae_igs:.2f}")
print(f"  RMSE: {rmse_igs:.2f}")
print()
print(f"GOPI vs MAGGIA:")
print(f"  MAE:  {mae_maggia:.2f}")
print(f"  RMSE: {rmse_maggia:.2f}")
print()
print(f"GOPI vs NAGOYA:")
print(f"  MAE:  {mae_nagoya:.2f}")
print(f"  RMSE: {rmse_nagoya:.2f}")
print("="*70)

results_december = enhanced_analysis(data)
print_enhanced_results(results_december, "COMPREHENSIVE ANALYSIS - December 1, 18, 19 and 22, 2024")

CONSOLIDATED RESULTS - December 1, 18, 19 and 22, 2024
(28 measurements: 7 stations × 4 times)
GOPI vs EMBRACE:
  MAE:  6.40
  RMSE: 8.15

GOPI vs IGS:
  MAE:  15.55
  RMSE: 16.18

GOPI vs MAGGIA:
  MAE:  18.11
  RMSE: 18.73

GOPI vs NAGOYA:
  MAE:  7.44
  RMSE: 9.20
COMPREHENSIVE ANALYSIS - December 1, 18, 19 and 22, 2024

EMBRACE:
  MAE:                   6.40
  RMSE:                  8.15
  Pearson Correlation:   0.472 (p=0.011)
  Taylor Skill Score:    0.720
  Kling-Gupta Efficiency: 0.453
    - Correlation (r):   0.472
    - Alpha (σp/σo):     0.860
    - Beta (μp/μo):      1.034

IGS:
  MAE:                   15.55
  RMSE:                  16.18
  Pearson Correlation:   0.835 (p=0.000)
  Taylor Skill Score:    0.902
  Kling-Gupta Efficiency: 0.697
    - Correlation (r):   0.835
    - Alpha (σp/σo):     0.875
    - Beta (μp/μo):      1.222

MAGGIA:
  MAE:                   18.11
  RMSE:                  18.73
  Pearson Correlation:   0.830 (p=0.000)
  Taylor Skill Score:    0.915


## December 18, 2024: 18:00 UT, 18:10 UT, 18:20 UT and 18:30 UT

In [ ]:
#The values ​​below refer, for each TEC source, consecutively, to the following times for December 18, 2024: 18:00 UT, 18:10 UT, 18:20 UT and 18:30 UT.

In [ ]:
import numpy as np

data = {
    'BRAZ': {
        'Gopi': [79.74, 80.90, 81.52, 81.67],
        'EMBRACE': [81.36, 82.63, 86.12, 89.26],
        'MAGGIA': [99.78, 100.54, 101.97, 97.66],
        'Nagoya': [82.49, 77.58, 86.32, 82.50]
    },
    'CUIB': {
        'Gopi': [74.03, 75.38, 76.57, 77.54],
        'EMBRACE': [76.98, 78.63, 78.75, 77.95],
        'MAGGIA': [94.45, 96.70, 98.65, 85.99],
        'Nagoya': [75.42, 77.14, 79.33, 80.47]
    },
    'NAUS': {
        'Gopi': [67.33, 67.10, 67.05, 67.06],
        'EMBRACE': [63.07, 62.57, 62.59, 62.60],
        'MAGGIA': [87.40, 94.99, 89.51, 87.51],
        'Nagoya': [61.57, 61.02, 60.88, 61.64]
    },
    'POAL': {
        'Gopi': [58.40, 57.57, 56.66, 55.93],
        'EMBRACE': [75.83, 76.34, 77.60, 74.33],
        'MAGGIA': [75.03, 75.28, 74.63, 75.09],
        'Nagoya': [74.34, 76.41, 77.51, 78.36]
    },
    'SALU': {
        'Gopi': [64.24, 64.09, 64.24, 64.72],
        'EMBRACE': [66.44, 65.81, 64.81, 64.78],
        'MAGGIA': [79.03, 79.40, 79.49, 77.81],
        'Nagoya': [66.20, 66.66, 67.51, 68.84]
    },
    'SAVO': {
        'Gopi': [90.05, 90.04, 89.57, 88.68],
        'EMBRACE': [83.04, 83.43, 83.02, 84.79],
        'MAGGIA': [99.74, 98.79, 98.48, 101.57],
        'Nagoya': [83.50, 77.60, 79.04, 78.62]
    },
    'SJSP': {
        'Gopi': [71.73, 73.19, 73.49, 72.30],
        'EMBRACE': [80.31, 80.79, 80.31, 82.70],
        'MAGGIA': [83.35, 84.76, 84.96, 84.29],
        'Nagoya': [83.00, 83.87, 81.58, 82.26]
    }
}

gopi_all = []
embrace_all = []
maggia_all = []
nagoya_all = []

for station in data.keys():
    gopi_all.extend(data[station]['Gopi'])
    embrace_all.extend(data[station]['EMBRACE'])
    maggia_all.extend(data[station]['MAGGIA'])
    nagoya_all.extend(data[station]['Nagoya'])

gopi_all = np.array(gopi_all)
embrace_all = np.array(embrace_all)
maggia_all = np.array(maggia_all)
nagoya_all = np.array(nagoya_all)

diff_embrace = gopi_all - embrace_all
diff_maggia = gopi_all - maggia_all
diff_nagoya = gopi_all - nagoya_all

mae_embrace = np.mean(np.abs(diff_embrace))
mae_maggia = np.mean(np.abs(diff_maggia))
mae_nagoya = np.mean(np.abs(diff_nagoya))

rmse_embrace = np.sqrt(np.mean(diff_embrace**2))
rmse_maggia = np.sqrt(np.mean(diff_maggia**2))
rmse_nagoya = np.sqrt(np.mean(diff_nagoya**2))

print("="*70)
print("CONSOLIDATED RESULTS - December 18, 2024")
print("(28 measurements: 7 stations × 4 times)")
print("="*70)
print(f"GOPI vs EMBRACE:")
print(f"  MAE:  {mae_embrace:.2f}")
print(f"  RMSE: {rmse_embrace:.2f}")
print()
print(f"GOPI vs MAGGIA:")
print(f"  MAE:  {mae_maggia:.2f}")
print(f"  RMSE: {rmse_maggia:.2f}")
print()
print(f"GOPI vs NAGOYA:")
print(f"  MAE:  {mae_nagoya:.2f}")
print(f"  RMSE: {rmse_nagoya:.2f}")
print("="*70)

results_december = enhanced_analysis(data)
print_enhanced_results(results_december, "COMPREHENSIVE ANALYSIS - December 18, 2024")

CONSOLIDATED RESULTS - December 18, 2024
(28 measurements: 7 stations × 4 times)
GOPI vs EMBRACE:
  MAE:  6.41
  RMSE: 8.59

GOPI vs MAGGIA:
  MAE:  16.29
  RMSE: 17.00

GOPI vs NAGOYA:
  MAE:  7.63
  RMSE: 9.64
COMPREHENSIVE ANALYSIS - December 18, 2024

EMBRACE:
  MAE:                   6.41
  RMSE:                  8.59
  Pearson Correlation:   0.645 (p=0.000)
  Taylor Skill Score:    0.787
  Kling-Gupta Efficiency: 0.595
    - Correlation (r):   0.645
    - Alpha (σp/σo):     0.810
    - Beta (μp/μo):      1.047

MAGGIA:
  MAE:                   16.29
  RMSE:                  17.00
  Pearson Correlation:   0.877 (p=0.000)
  Taylor Skill Score:    0.934
  Kling-Gupta Efficiency: 0.735
    - Correlation (r):   0.877
    - Alpha (σp/σo):     0.931
    - Beta (μp/μo):      1.225

NAGOYA:
  MAE:                   7.63
  RMSE:                  9.64
  Pearson Correlation:   0.491 (p=0.008)
  Taylor Skill Score:    0.694
  Kling-Gupta Efficiency: 0.437
    - Correlation (r):   0.491
    - 